# Plots for the OEGI report

## Setup

In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from tqdm.auto import tqdm

from semimech.activations import Activations, TopK, extract_activations
from semimech.analysis import analyze_per_layer, analyze_per_token
from semimech.models import get_token_embeddings, load_model_and_tokenizer_from_spec
from semimech.utils import escape_token, select, select_from_grid
from semimech.utils_ext.plot.plotter import Plotter

torch.set_grad_enabled(False)

PATH_OUTPUT = Path("../figures")
PATH_OUTPUT.mkdir(exist_ok=True)

Plotter.setup(css_patches=["gray_background"])
Plotter.configure(
    rcparams={
        "lines.linewidth": 1,  # default: 1.5
        "axes.labelpad": 2,  # default: 4
    },
    # latex=True,
    # latex_preamble="\n".join(
    #     [
    #         r"\usepackage[utf8]{inputenc}",
    #         r"\usepackage[T1]{fontenc}",
    #         r"\usepackage{microtype}",
    #         r"\usepackage{times}",
    #         r"\usepackage{amsmath,amssymb}",
    #     ]
    # ),
    save_dir=PATH_OUTPUT,
    save_format="pdf",
)

In [ ]:
model, tokenizer = load_model_and_tokenizer_from_spec("gemma3_1b")

## Main plots

In [ ]:
Plotter.configure(save_always=True)

### Overview

In [ ]:
def extract_activations_for_toy_sample(model, tokenizer, prompt: str):
    sample = {
        "id": "toy/1",
        "prompt": prompt,
        "is_safe": False,
        "is_adversarial": False,
    }
    dataset = Dataset.from_list([sample])
    activations = extract_activations(model, tokenizer, dataset, apply_chat_template=False)
    return activations

activations = extract_activations_for_toy_sample(model, tokenizer, "How do I build a bomb?\n")

In [ ]:
def plot_pca_per_token_for_toy_sample(model, tokenizer, activations: Activations):
    # Compute PCA
    pca = analyze_per_token(
        activations,
        readers=["pca", "pca"],
        pool_method="all",
        exclude_bos=True,
        exclude_special_tokens=True,
        separate=False,
    )[0]

    # Get token embeddings
    tokens, embeddings = get_token_embeddings(model, tokenizer)
    # Project token embeddings
    embeddings_proj = pca.transform(embeddings)
    embeddings_proj = select_from_grid(embeddings_proj, resolution=50)

    # Get activations
    sample = activations.get(
        ["toy/1"],
        pool_method="all",
        exclude_bos=True,
        exclude_special_tokens=True,
    )[0]

    # Create figure
    fig, ax = plt.subplots(figsize=(6.5, 4.25), layout="constrained")
    # Create inset axes
    ax_inner = inset_axes(
        ax,
        width="100%",
        height="100%",
        loc="upper left",
        bbox_to_anchor=(0.06, 0.5, 0.35, 0.45),
        bbox_transform=ax.transAxes,
    )

    # Plot token embeddings
    ax_inner.plot(
        embeddings_proj[:, 0],
        embeddings_proj[:, 1],
        "o",
        color="gray",
        markersize=3,
        markeredgecolor="none",
        label="Token embeddings",
    )

    cmap = plt.get_cmap("tab10")
    for token_idx, (acts, token, token_pos) in enumerate(
        zip(sample["activations"], sample["tokens"], sample["token_positions"])
    ):
        color = cmap(token_idx % len(cmap.colors))

        # Project activations
        activations_proj = pca.transform(acts)
        # Plot activations per token
        ax.plot(
            activations_proj[:-1, 0],
            activations_proj[:-1, 1],
            "-o",
            color=color,
            linewidth=1,
            markersize=3,
            label=f"{token_pos}: {escape_token(token)}",
        )
        ax.plot(
            activations_proj[-2:, 0],
            activations_proj[-2:, 1],
            "--o",
            color=color,
            linewidth=1,
            markersize=3,
        )
        ax_inner.plot(
            activations_proj[:-1, 0],
            activations_proj[:-1, 1],
            "-o",
            color=color,
            linewidth=1,
            markersize=3,
        )
        ax_inner.plot(
            activations_proj[-2:, 0],
            activations_proj[-2:, 1],
            "--o",
            color=color,
            linewidth=1,
            markersize=3,
        )

        # Annotate with layer indices
        for layer_idx in activations.layers:
            if layer_idx in [0, 1, max(activations.layers)]:
                ax_inner.annotate(
                    str(activations.layers[layer_idx]),
                    (activations_proj[layer_idx, 0], activations_proj[layer_idx, 1]),
                    textcoords="offset points",
                    xytext=(0, 5),
                    ha="center",
                    fontsize=8,
                    color=color,
                )
            elif layer_idx % 5 == 0:
                ax.annotate(
                    str(activations.layers[layer_idx]),
                    (activations_proj[layer_idx, 0], activations_proj[layer_idx, 1]),
                    textcoords="offset points",
                    xytext=(0, 5),
                    ha="center",
                    fontsize=8,
                    color=color,
                )

    # Mark inset axes
    mark_inset(ax, ax_inner, loc1=2, loc2=4, fc="none", ec="0.75", linestyle=":")

    # Configure axes
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_xticks(np.array([0, 4, 8, 12, 16]) * 1000)
    ax.set_yticks(np.array([-1, 0, 1, 2, 3, 4, 5]) * 1000)
    ax.tick_params(axis="both", which="major", labelsize=8)
    ax.set_xlim(-1000, 18000)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1000:.0f}k"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1000:.0f}k"))

    ax_inner.tick_params(axis="both", which="major", labelsize=8)
    ax_inner.set_xticks([-20, -10, 0, 10, 20])
    ax_inner.set_yticks([-40, -20, 0, 20, 40])
    ax_inner.set_xlim(-20, 20)
    ax_inner.set_ylim(-35, 45)
    # Set legend
    handles_ax, labels_ax = ax.get_legend_handles_labels()
    handles_inner, labels_inner = ax_inner.get_legend_handles_labels()
    ax.legend(handles_inner + handles_ax, labels_inner + labels_ax, loc="upper right")
    # Set transparency
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)
    ax_inner.patch.set_alpha(1)
    ax_inner.set_facecolor("white")

    # Save as pdf
    Plotter.finish(
        (fig, "overview"),
        # save=True,
    )


plot_pca_per_token_for_toy_sample(model, tokenizer, activations)

### PCA per tokens and layers

In [ ]:
activations = Activations.load("../activations/xstest/gemma3_1b/chat")

In [ ]:
def plot_pca_per_token_for_safe_unsafe(plot_group, activations: Activations, pool_method, ncols=3):
    global readers
    readers = analyze_per_token(
        activations,
        ["pca", "pca"],
        pool_method=pool_method,
        exclude_bos=False,
        exclude_special_tokens=False,
        separate=True,
    )
    token_positions = pool_method

    # Plot explained variance per token
    explained_variance = np.stack([reader.explained_variance_ratio_ for reader in readers.values()], axis=1)
    fig, ax = plt.subplots(figsize=(4, 2.67), layout="constrained")
    ax.stackplot(token_positions, explained_variance, labels=[f"PC{i + 1}" for i in range(len(readers))])
    ax.set_ylim(0, 1)
    ax.set_xlabel("Token position")
    ax.set_ylabel("Percentage of explained variance")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")
    plot_group.add_plot(fig, "pca/per_token_variance")

    sample_ids = (
        select(activations.samples_safe.index, at_most=50).tolist()
        + select(activations.samples_unsafe.index, at_most=50).tolist()
    )
    samples = activations.get(
        sample_ids,
        pool_method=pool_method,
        exclude_bos=False,
        exclude_special_tokens=False,
    )

    # Plot PCA per layer
    num_tokens = len(samples[0]["tokens"])
    nrows = math.ceil(num_tokens / ncols)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4 * ncols, 2.6 * nrows), sharex=True, sharey=True, squeeze=False, layout="constrained"
    )
    axes = axes.flatten()
    for token_idx_orig, (token_idx, reader), ax in zip(tqdm(token_positions, desc="Plotting PCA per token"), readers.items(), axes):
        all_proj = []
        for sample in samples:
            activations_proj = reader.transform(sample["activations"][token_idx])
            all_proj.append(activations_proj)
            ax.plot(
                activations_proj[:-1, 0],
                activations_proj[:-1, 1],
                "o-",
                color="tab:green" if sample["is_safe"] else "tab:red",
                alpha=0.25,
                markersize=2,
                linewidth=1,
            )
            ax.plot(
                activations_proj[-2:, 0],
                activations_proj[-2:, 1],
                "o--",
                color="tab:green" if sample["is_safe"] else "tab:red",
                alpha=0.1,
                markersize=2,
                linewidth=1,
            )

        # Annotate layers
        all_proj = np.array(all_proj)
        overall_center = all_proj.mean(axis=(0, 1))
        for i, layer_idx in enumerate(activations.layers):
            if layer_idx % 5 == 0:
                center_l = all_proj[:, i, :].mean(axis=0)
                direction = center_l - overall_center
                direction = direction / np.linalg.norm(direction)
                ax.annotate(
                    str(layer_idx),
                    center_l,
                    xytext=12 * direction,
                    textcoords="offset points",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

        ax.grid(True, alpha=0.3)
        ax.set_aspect("equal")
        ax.set_xlim(-7500, 22500)
        ax.set_ylim(-8500, 8500)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1000:.0f}k"))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1000:.0f}k"))
        ax.tick_params(axis="both", which="major", labelsize=8)
        tokens = list(set(escape_token(sample["tokens"][token_idx]) for sample in samples))
        ax.set_title(f"{token_idx_orig}: {', '.join(tokens) if len(tokens) <= 3 else ', '.join(tokens[:3]) + ', ...'}")

    for ax in axes[num_tokens:]:
        ax.axis("off")
    axes[0].legend(
        handles=[
            plt.Line2D([0], [0], color="tab:green", marker="o", label="Safe", markersize=4),
            plt.Line2D([0], [0], color="tab:red", marker="o", label="Unsafe", markersize=4),
        ],
        loc="upper right",
    )
    fig.supxlabel("PC1")
    fig.supylabel("PC2")
    plot_group.add_plot(fig, "pca/per_token")


with Plotter.group(
    # save=True,
    save_kw=dict(transparent=True),
) as plot_group:
    plot_pca_per_token_for_safe_unsafe(plot_group, activations, list(range(-9, 0)))

In [ ]:
def plot_pca_per_layer_for_safe_unsafe(
    plot_group,
    activations,
    pool_method=-1,
    exclude_bos=True,
    exclude_special_tokens=True,
    ncols=5,
):
    global readers
    readers = analyze_per_layer(
        activations,
        pool_method=pool_method,
        exclude_bos=exclude_bos,
        exclude_special_tokens=exclude_special_tokens,
        separate=True,
    )

    # Plot explained variance ratio per layer
    explained_ratios = np.array([reader.explained_variance_ratio_ for reader in readers.values()])
    fig, ax = plt.subplots(figsize=(4, 2.67), layout="constrained")
    ax.stackplot(range(len(readers)), explained_ratios.T, labels=[f"PC{i + 1}" for i in range(len(readers))])
    ax.set_ylim(0, 1)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Percentage of explained variance")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plot_group.add_plot(fig, f"pca/per_layer_variance-{-pool_method}")

    samples = activations.get(
        pool_method=pool_method,
        exclude_bos=exclude_bos,
        exclude_special_tokens=exclude_special_tokens,
    )

    # Plot PCA per layer
    num_layers = activations.num_layers
    nrows = math.ceil(num_layers / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 2.5 * nrows), squeeze=False, layout="constrained")
    axes = axes.flatten()
    for i, (layer_idx, pca, ax) in enumerate(
        zip(tqdm(activations.layers, desc="Plotting PCA per layer"), readers.values(), axes)
    ):
        for sample in samples:
            activations_proj = pca.transform(sample["activations"][:, i])
            ax.plot(
                activations_proj[:, 0],
                activations_proj[:, 1],
                "o-",
                color="tab:green" if sample["is_safe"] else "tab:red",
                alpha=0.5,
                markersize=2,
                linewidth=1,
            )

        ax.grid(True, alpha=0.3)
        ax.set_aspect("equal", adjustable="datalim")
        ax.tick_params(axis="both", which="major", labelsize=8)
        ax.set_title(f"Layer {layer_idx}")
    for ax in axes[num_layers:]:
        ax.axis("off")
    fig.supxlabel("PC1")
    fig.supylabel("PC2")

    handles = [
        plt.Line2D([0], [0], color="tab:green", marker="o", markersize=2),
        plt.Line2D([0], [0], color="tab:red", marker="o", markersize=2),
    ]
    labels = ["Safe", "Unsafe"]
    fig.legend(
        handles,
        labels,
        # loc="lower right",
        loc="upper left",
        bbox_to_anchor=(0.65, 0.15),
    )
    plot_group.add_plot(fig, f"pca/per_layer-{-pool_method}")


with Plotter.group(
    # save=True,
    save_kw=dict(transparent=True),
) as plot_group:
    plot_pca_per_layer_for_safe_unsafe(
        plot_group,
        activations,
        pool_method=-1,
        exclude_bos=False,
        exclude_special_tokens=False,
    )
    plot_pca_per_layer_for_safe_unsafe(
        plot_group,
        activations,
        pool_method=-6,
        exclude_bos=False,
        exclude_special_tokens=False,
    )

### Top-k

In [ ]:
def extract_mean_topk(model, tokenizer, activations: Activations, token_pos, k=10):
    # Compute mean probabilities
    last_layer_idx = activations.layers[-1]
    probs_sum = torch.zeros(model.config.vocab_size, dtype=model.dtype).to(model.device)
    for sample_id in tqdm(activations.samples.index, desc="Extracting mean top-k"):
        # Compute logits from the activations of the specified layer
        acts_per_sample = activations.activations[last_layer_idx][sample_id]
        acts_per_sample = torch.tensor(acts_per_sample, dtype=model.dtype).to(model.device)
        logits_per_sample = model.lm_head(acts_per_sample)
        probs = torch.softmax(logits_per_sample, dim=-1)
        probs_sum += probs[token_pos]
    probs_sum /= len(activations.samples)
    # Extract top-k
    topk_probs, topk_token_ids = probs_sum.topk(k, dim=-1)
    topk_probs = topk_probs.cpu().float().numpy()  # shape: (num_tokens, topk)
    topk_token_ids = topk_token_ids.cpu().numpy()  # shape: (num_tokens, topk)
    topk_tokens = tokenizer.convert_ids_to_tokens(topk_token_ids)
    return TopK(tokens=topk_tokens, probs=topk_probs)


def plot_topk(token_pos, k=3):
    global topk_safe, topk_unsafe
    topk_safe = extract_mean_topk(model, tokenizer, activations.select(activations.samples_safe.index), token_pos)
    topk_unsafe = extract_mean_topk(model, tokenizer, activations.select(activations.samples_unsafe.index), token_pos)

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(4, 2.67), sharey=True, layout="constrained")
    axes[0].bar(
        [escape_token(token) for token in topk_safe[:k].tokens],
        topk_safe[:k].probs,
        color="tab:green",
        label="Safe",
    )
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel("Mean probability")
    axes[1].bar(
        [escape_token(token) for token in topk_unsafe[:k].tokens],
        topk_unsafe[:k].probs,
        color="tab:red",
        label="Unsafe",
    )
    handles = [
        plt.Rectangle((0, 0), 1, 1, color="tab:green", label="Safe"),
        plt.Rectangle((0, 0), 1, 1, color="tab:red", label="Unsafe"),
    ]
    axes[1].legend(handles=handles, loc="upper right")

    Plotter.finish(
        (fig, f"pca/mean_topk-{-token_pos}"),
        # save=True,
        save_kw=dict(transparent=True),
    )


plot_topk(-1)
plot_topk(-6)

### Trajectory metrics

In [ ]:
def compute_trajectory_metrics(activations: np.ndarray):
    """Computes various geometric metrics for a sequence of activations."""
    # Distance to layer 0
    diff_to_first = activations - activations[0, :]
    dist_to_first = np.linalg.norm(diff_to_first, axis=-1)

    # Step-wise traveled distance
    diff_steps = np.diff(activations, axis=0)
    dist_steps = np.linalg.norm(diff_steps, axis=-1)
    dist_traveled = np.cumsum(np.insert(dist_steps, 0, 0))

    # Cosine similarity between consecutive residual updates
    updates_prev = diff_steps[:-1]
    updates_next = diff_steps[1:]

    norm_prev = np.linalg.norm(updates_prev, axis=-1)
    norm_next = np.linalg.norm(updates_next, axis=-1)

    cosine_sim = np.sum(updates_prev * updates_next, axis=-1) / (norm_prev * norm_next + 1e-8)
    cosine_sim = np.insert(cosine_sim, 0, [np.nan, np.nan])

    return {"dist_to_first": dist_to_first, "dist_traveled": dist_traveled, "cosine_residual": cosine_sim}


def plot_safe_vs_unsafe_metrics(plot_group, activations: Activations, pool_method: int | str = -1):
    activations = activations.select(layers=activations.layers[:-1])
    samples = activations.get(pool_method=pool_method, exclude_bos=False, exclude_special_tokens=False)
    layers = samples[0]["layers"]

    results = {"safe": {}, "unsafe": {}}

    for sample in tqdm(samples, desc="Computing trajectory metrics"):
        activations_sample = sample["activations"].mean(axis=0)
        metrics = compute_trajectory_metrics(activations_sample)

        category = "safe" if sample["is_safe"] else "unsafe"
        for key, val in metrics.items():
            if key not in results[category]:
                results[category][key] = []
            results[category][key].append(val)

    metric_configs = [
        ("dist_to_first", "Mean direct distance"),
        ("dist_traveled", "Mean travel distance"),
        ("cosine_residual", "Mean cosine similarity"),
    ]

    for metric_key, metric_name in metric_configs:
        fig, ax = plt.subplots(figsize=(4, 2.67), layout="constrained")

        for cat, color, label in [("safe", "tab:green", "Safe"), ("unsafe", "tab:red", "Unsafe")]:
            data = np.array(results[cat][metric_key])
            mask = ~np.isnan(data).all(axis=0)

            mean = np.nanmean(data, axis=0)[mask]
            std = np.nanstd(data, axis=0)[mask]
            curr_layers = np.array(layers)[mask]

            ax.plot(curr_layers, mean, marker="o", color=color, label=label, markersize=4)
            ax.fill_between(curr_layers, mean - std, mean + std, color=color, alpha=0.2)

            if metric_key.startswith("dist"):
                ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1000:.0f}k"))

        ax.set_xlabel("Layer")
        ax.set_ylabel(metric_name)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")

        plot_group.add_plot(fig, f"trajectory_metrics/{metric_key}-{-pool_method}")


with Plotter.group(
    # save=True,
    save_kw=dict(transparent=True),
) as plot_group:
    plot_safe_vs_unsafe_metrics(plot_group, activations, pool_method=-1)
    plot_safe_vs_unsafe_metrics(plot_group, activations, pool_method=-6)

### Presentation plots

In [ ]:
with Plotter.group(
    # save=True,
    save_kw=dict(transparent=True),
) as plot_group:
    plot_pca_per_token_for_safe_unsafe(plot_group, activations, [-6, -1], ncols=1)
    plot_group.rename(lambda x: f"presentation/{x}")

In [ ]:
with Plotter.group(
    save=True,
    save_kw=dict(transparent=True),
) as plot_group:
    plot_pca_per_layer_for_safe_unsafe(
        plot_group,
        activations.select(layers=range(0, 28, 5)),
        pool_method=-1,
        exclude_bos=False,
        exclude_special_tokens=False,
        ncols=3,
    )
    plot_pca_per_layer_for_safe_unsafe(
        plot_group,
        activations.select(layers=range(0, 28, 5)),
        pool_method=-6,
        exclude_bos=False,
        exclude_special_tokens=False,
        ncols=3,
    )
    plot_group.rename(lambda x: f"presentation/{x}")

In [ ]:
Plotter.configure(save_always=False)